# Training a Surrogate Model with PyTorch

This notebook illustrates how to train a simple neural network surrogate model using Pytorch. In this notebook we look at the simple case of the `arnett_with_features` function from redback. We show how to generate the training data, fit the model, and evaluate the model. This notebook serves as a illustration and proof of concept. Each source model (type of phenomena) will likely need a
tailored ML architecture and training procedure to get the best results.

This notebook uses multiple packages that are not installed with redback_surrogates by default (and must be install manually), including:

   * redback (for the arnett model)
   * torch (for the training)

In [ ]:
import numpy as np

from astropy.table import Table
from redback.transient_models.supernova_models import arnett_with_features
from tqdm import tqdm
from scipy.stats import loguniform

from redback_surrogates.learned_surrogate_data import LearnedSurrogateDataset
from redback_surrogates.learned_surrogate_train import evaluate_learned_model, train_pytorch_model

## Creating the Training Data

We start with a simple method to create the training data. This consists of three steps:
   1) We generate all of the input parameters needed by the function.
   2) Generate an output grid (spectra) for each of the sets of parameters.
   3) Wrap the data in a `LearnedSurrogateDataset`.

We put these steps in a helper function so we can use the same approach for generating the test data.

In [ ]:
def generate_arnett_data(num_samples):
    """Generate training/validation data for the Arnett model with
    randomly sampled parameters and outputs as the spectra.

    :param num_samples: Number of samples to generate.
    """
    times = np.array([0.0, 1.0])  # Unused by "spectra" output type

    # Generate the parameter samples. For the simplicity of the example,
    # we use much smaller ranges than the full priors provided by redback.
    data = {
        "redshift": np.random.uniform(0.1, 0.2, size=num_samples),
        "f_nickel": np.random.uniform(0.1, 1.0, size=num_samples),
        "mej": np.random.uniform(1.0, 10.0, size=num_samples),
        "vej": np.random.uniform(1_000, 2_000, size=num_samples),
        "kappa": np.random.uniform(0.05, 0.15, size=num_samples),
        "kappa_gamma": np.random.uniform(0.025, 0.035, size=num_samples),
        "temperature_floor": np.random.uniform(4000.0, 6000.0, size=num_samples),
    }
    
    # Compute the output for each sample.
    output_grids = []
    out_times = None
    out_lambdas = None
    for i in tqdm(range(num_samples)):
        grid = arnett_with_features(
            time=times,
            redshift=data["redshift"][i],
            f_nickel=data["f_nickel"][i],
            mej=data["mej"][i],
            vej=data["vej"][i],
            kappa=data["kappa"][i],
            kappa_gamma=data["kappa_gamma"][i],
            temperature_floor=data["temperature_floor"][i],
            output_format="spectra",
        )
        output_grids.append(grid.spectra)

        # Save the time and wavelength grid from the first sample.
        if i == 0:
            out_times = grid.time
            out_lambdas = grid.lambdas

    data["grid"] = np.array(output_grids)

    results = LearnedSurrogateDataset(
        Table(data),
        output_column="grid",
        times=out_times,
        wavelengths=out_lambdas,
    )
    return results

Now we create the actual training data with 1,000 samples.

In [ ]:
training_data = generate_arnett_data(num_samples=500)

We can look at the plot of light curve over the wavelengths and times to get an idea of what we are trying to fit.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

# Get the data.
x = np.log(training_data.times)
y = np.log(training_data.wavelengths)
z = training_data.data["grid"][0]
mesh_x, mesh_y = np.meshgrid(x, y, indexing='ij')

# Plot the surface.
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
surf = ax.plot_surface(
    mesh_x,
    mesh_y,
    z,
    cmap=mpl.cm.coolwarm,
    linewidth=0,
)
ax.set_xlabel('Log Time (days)')
ax.set_ylabel('Log Wave (A)')
plt.show()

# Print out the bounds on the output, since the z-axis is not showing them correctly.
print(f"Output min: {np.min(z)}")
print(f"Output max: {np.max(z)}")

## Training the Model

Users can pick their favorite machine learning platform to train models. Here we use the included `train_pytorch_model` function, which trains a multi-layer neural network. The input layer will correspond to the parameters. The output layer will correspond to the SED grid as defined by the function (in this case 3000 time steps and 100 wavelengths). We use hidden layers of size 32, 16, and 64.

In [ ]:
model = train_pytorch_model(
    training_data,
    hidden_sizes=[32, 16, 64],
    training_epochs=100,
    learning_rate=0.01,
    verbose=True,
)

## Evaluating the Model

The returned object is a `LearnedSurrogateModel` that can be saved to disk as a ONNX model and used in redback. We can generate a new (random) test set and evaluate the performance of the model on that set.

In [ ]:
test_data = generate_arnett_data(num_samples=200)

rmse = evaluate_learned_model(model, test_data)
print(f"Average Test RMSE: {rmse}")

We can also evaluate the model at the predefined times/wavelengths and look at the resulting plot. We use the first training point to match the plot above.

In [ ]:
input_params = training_data.get_input_dict(0)
output = model.predict_spectra_grid(**input_params)

# Plot the surface.
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
surf = ax.plot_surface(
    mesh_x,  # Same mesh as training data
    mesh_y,  # Same mesh as training data
    output,
    cmap=mpl.cm.coolwarm,
    linewidth=0,
)
ax.set_xlabel('Log Time (days)')
ax.set_ylabel('Log Wavelength (Angstrom)')
plt.show()

# Print out the bounds on the output, since the z-axis is not showing them correctly.
print(f"Output min: {np.min(output)}")
print(f"Output max: {np.max(output)}")

We can also look at the error surface.

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

output_err = output - z

surf = ax.plot_surface(
    mesh_x,  # Same mesh as training data
    mesh_y,  # Same mesh as training data
    output_err,
    cmap=mpl.cm.coolwarm,
    linewidth=0,
)
ax.set_xlabel('Log Time (days)')
ax.set_ylabel('Log Wavelength (Angstrom)')
plt.show()

# Print out the bounds on the output, since the z-axis is not showing them correctly.
print(f"Output min: {np.min(output_err)}")
print(f"Output max: {np.max(output_err)}")

## Improving Training

As noted in the introduction, this notebook provides a basic proof of concept. From the predicted values, we can see the model still has a ways to go. Part of this might be due to model structure. Another part is due to the limited space of training data. To keep memory consumption reasonable, we only use 1,000 examples. We can only cover part of the input parameter space in such a limited number of samples.

The example script `examples\example_arnett_training.py` provides a more comprehensive training approach where new batches of training data are regularly generated during the training process. This allows the model to train on a greater variety of training data while keeping the total memory required reasonable.

We expect that almost all models will need customized training beyond what `train_pytorch_model` provides.